<a href="https://colab.research.google.com/github/Bukunmi2108/ml-journey/blob/main/research/p2/quant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Quantization from scratch

In [83]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [84]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [85]:
class INTQuantizer():
  @staticmethod
  def quantize_int8_per_tensor(tensor: torch.Tensor):
    scale = tensor.abs().max() / 127
    scale = scale.clamp(min=1e-8)
    out = torch.round(tensor / scale).clamp(-128, 127).to(torch.int8)
    return out, scale

  @staticmethod
  def dequantize_int8_per_tensor(q_tensor: torch.Tensor, scale: torch.Tensor):
    return q_tensor.float() * scale

  @staticmethod
  def quantize_int8_per_channel(tensor: torch.Tensor): # quantizes/scales per row
    scales = tensor.abs().amax(dim=1, keepdim=True) / 127
    scales = scales.clamp(min=1e-8)
    out = torch.round(tensor / scales).clamp(-127, 127).to(torch.int8)
    return out, scales

  @staticmethod
  def dequantize_int8_per_channel(q_tensor: torch.Tensor, scales: torch.Tensor):
    return q_tensor.float() * scales

  @staticmethod
  def quantize_int4_groupwise(tensor: torch.Tensor, group_size = 128):
    out_dim, in_dim = tensor.shape
    pad = (group_size - (in_dim % group_size)) % group_size
    if pad:
      T_padded = F.pad(tensor, (0, pad))
    else:
      T_padded = tensor
    padded_dim = T_padded.shape[1]
    T_reshaped = T_padded.view(out_dim, padded_dim // group_size, group_size)

    scales = T_reshaped.abs().amax(dim=2, keepdim=True) / 7
    scales = scales.clamp(min=1e-8)
    out = torch.round(T_reshaped / scales).clamp(-8, 7).to(torch.int8)

    meta = {
        "original_shape": tensor.shape,
        "padded_shape": T_padded.shape,
        "group_size": group_size,
        "pad": pad
    }

    return out, scales, meta

  @staticmethod
  def dequantize_int4_groupwise(q_tensor: torch.Tensor, scales: torch.Tensor, meta: dict):
    T_padded_hat = (q_tensor.float() * scales).view(meta["padded_shape"])
    out_dim, in_dim = meta["original_shape"]
    return T_padded_hat[:, :in_dim]

  @staticmethod
  def tensor_error(tensor: torch.Tensor, tensor_hat: torch.Tensor):
    mse = torch.mean((tensor - tensor_hat) ** 2).item()
    mae = torch.mean(torch.abs(tensor - tensor_hat)).item()
    max_err = torch.max(torch.abs(tensor -tensor_hat)).item()
    rel_mse = mse / (torch.mean(tensor ** 2).item() + 1e-12)
    return mse, mae, max_err, rel_mse

Test INTQuantizer class

In [86]:
iq = INTQuantizer()
test_tensor = torch.tensor([-10.0, 0.0, 5.0, 10.0])

quantized_tensor, scale = iq.quantize_int8_per_tensor(test_tensor)
assert torch.isclose(scale, torch.tensor(10.0/127.0))

dequantized_tensor = iq.dequantize_int8_per_tensor(quantized_tensor, scale)
assert torch.allclose(test_tensor, dequantized_tensor, atol=scale.item()/2)

In [87]:
test_tensor = torch.tensor([[-10., 0., 5., 10.], [-20., -5., 0., 15.]])
quantized_tensor, scales = iq.quantize_int8_per_channel(test_tensor)
dequantized_tensor = iq.dequantize_int8_per_channel(quantized_tensor, scales)
max_atol = scales.max().item() / 2
assert torch.allclose(test_tensor, dequantized_tensor, atol=max_atol)

In [88]:
test_tensor_groupwise = torch.randn(10, 256) * 10
group_size = 64

quantized_tensor_groupwise, scales_groupwise, meta_groupwise = iq.quantize_int4_groupwise(test_tensor_groupwise, group_size=group_size)
dequantized_tensor_groupwise = iq.dequantize_int4_groupwise(quantized_tensor_groupwise, scales_groupwise, meta_groupwise)
max_atol_groupwise = scales_groupwise.max().item() / 2
assert torch.allclose(test_tensor_groupwise, dequantized_tensor_groupwise, atol=max_atol_groupwise)